In [ ]:
! pip install transformers accelerate torch

Seed data fetching

In [ ]:
from datasets import load_dataset
import html
import json
import random
import re
import time
import requests

random.seed(42)

MIN_WORDS, MAX_WORDS = 50, 100
N_PER_SOURCE = 50


def clean_text(value):
    """Normalize markup and whitespace before evaluating seed quality."""
    text = html.unescape(str(value or ""))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_clean_swedish_sample(text, min_words=MIN_WORDS, max_words=MAX_WORDS):
    """Reject boilerplate, link-heavy, repetitive, and non-prose samples."""
    words = re.findall(r"[A-Za-zÅÄÖåäö]+(?:-[A-Za-zÅÄÖåäö]+)?", text.lower())
    if not min_words <= len(words) <= max_words:
        return False
    if len(words) < 20 or len(set(words)) / len(words) < 0.45:
        return False
    if len(re.findall(r"[A-Za-zÅÄÖåäö]", text)) / max(len(text), 1) < 0.55:
        return False
    boilerplate = ("cookie", "javascript", "alla rättigheter", "lorem ipsum")
    return not any(phrase in text.lower() for phrase in boilerplate)


def reservoir_sample_from_stream(stream, source_name, n_per_source, overscan_factor=200,
                                 text_key="text", title_key="title", id_key="id"):
    sampled, seen = [], 0
    for row in stream:
        text = clean_text(row.get(text_key, ""))
        if not is_clean_swedish_sample(text):
            continue
        seen += 1
        item = {
            "id": f"{source_name}:{row.get(id_key, seen)}",
            "title": clean_text(row.get(title_key, "")),
            "text": text,
            "source": source_name,
        }
        if len(sampled) < n_per_source:
            sampled.append(item)
        else:
            replacement_index = random.randint(0, seen - 1)
            if replacement_index < n_per_source:
                sampled[replacement_index] = item
        if seen >= n_per_source * overscan_factor:
            break
    print(f"{source_name}: sampled {len(sampled)} clean documents (scanned {seen} candidates)")
    return sampled


# Encyclopedic, literary, and web-register samples.
wiki_stream = load_dataset("wikimedia/wikipedia", "20231101.sv", split="train", streaming=True)
wiki_samples = reservoir_sample_from_stream(wiki_stream, "wikipedia", N_PER_SOURCE)

wikisource_stream = load_dataset("wikimedia/wikisource", "20231201.sv", split="train", streaming=True)
wikisource_samples = reservoir_sample_from_stream(wikisource_stream, "wikisource", N_PER_SOURCE)

mc4_stream = load_dataset("allenai/c4", "sv", split="train", streaming=True)
mc4_samples = reservoir_sample_from_stream(mc4_stream, "mc4_web", N_PER_SOURCE)


def fetch_riksdagen_docs(n_docs, doktyp="mot", max_pages=30):
    samples = []
    base_url = "https://data.riksdagen.se/dokumentlista/"
    for page in range(1, max_pages + 1):
        params = {"doktyp": doktyp, "sort": "datum", "sortorder": "desc", "utformat": "json", "p": page}
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        documents = response.json().get("dokumentlista", {}).get("dokument", [])
        if not documents:
            break
        for document in documents:
            text_url = document.get("dokument_url_text")
            if not text_url:
                continue
            try:
                text = clean_text(requests.get(text_url, timeout=30).text)
            except requests.RequestException:
                continue
            if is_clean_swedish_sample(text):
                samples.append({
                    "id": f"riksdagen:{document.get('id', len(samples) + 1)}",
                    "title": clean_text(document.get("titel", "")),
                    "text": text,
                    "source": "riksdagen",
                })
                if len(samples) >= n_docs:
                    return samples
            time.sleep(0.2)
    print(f"riksdagen: sampled {len(samples)} clean documents")
    return samples


riksdagen_samples = fetch_riksdagen_docs(N_PER_SOURCE)

all_samples = wiki_samples + wikisource_samples + mc4_samples + riksdagen_samples
seen_texts, deduped = set(), []
for item in all_samples:
    fingerprint = re.sub(r"\W+", "", item["text"].lower())[:500]
    if fingerprint and fingerprint not in seen_texts:
        seen_texts.add(fingerprint)
        deduped.append(item)

random.shuffle(deduped)
with open("seed_sample_combined.jsonl", "w", encoding="utf-8") as f:
    for row in deduped:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(deduped)} clean, deduplicated seed samples to seed_sample_combined.jsonl")


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

wikipedia: sampled 50 clean documents (scanned 10000 candidates)


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

wikisource: sampled 50 clean documents (scanned 758 candidates)


README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

mc4_web: sampled 50 clean documents (scanned 10000 candidates)
riksdagen: sampled 0 clean documents
Saved 150 clean, deduplicated seed samples to seed_sample_combined.jsonl


Below code is for generation, evaluation and refinement

In [1]:
# One-cell pipeline: Qwen 3.5 generates/refines; Qwen 2.5 evaluates.
# Seed-data preparation remains in the separate cell above.

import gc
import json
import os
import re

import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

GENERATOR_MODEL_ID = "Qwen/Qwen3.5-4B"
# JUDGE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" # new model
SEED_INPUT_PATH = "/content/seed_sample_combined.jsonl"
INITIAL_OUTPUT_PATH = "initial_file_instruction.jsonl"
REGENERATED_OUTPUT_TEMPLATE = "regenerated_instruction_{attempt}.jsonl"
SCORE_THRESHOLD = 3
MAX_REGENERATION_ATTEMPTS = 3
DTYPE = torch.float16

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab: Runtime -> Change runtime type -> GPU.")


INITIAL_PROMPT = """Utgå från den svenska källtexten nedan och skriv EN enda naturlig svensk fråga eller instruktion som texten kan vara ett korrekt svar på.

[KÄLLTEXT]
{seed_doc}

Hårda formatkrav:
- Svara med exakt en rad: endast den färdiga frågan eller instruktionen.
- Ingen analys, motivering, rubrik, punktlista, citattecken eller etikett.
- Skriv inte "Thinking Process", "Analys", "Svar" eller liknande.
- Använd idiomatisk svenska och lägg inte till fakta som saknas i källtexten.
"""

REFINEMENT_PROMPT = """Skriv om den tidigare svenska instruktionen så att den åtgärdar utvärderarens återkoppling.

[KÄLLTEXT]
{seed_doc}

[TIDIGARE INSTRUKTION]
{instruction}

[UTVÄRDERARENS ÅTERKOPPLING]
{eval_reasoning}

Svara med exakt en rad och endast den förbättrade frågan eller instruktionen på idiomatisk svenska.
Ingen analys, motivering, rubrik, punktlista eller etikett. Lägg inte till fakta som saknas i källtexten.
"""

JUDGE_PROMPT = """Du är en sträng kvalitetsgranskare av svenska instruktioner.

[KÄLLTEXT / FÖRVÄNTAT SVAR]
{seed_doc}

[GENERERAD INSTRUKTION]
{instruction}

Bedöm på en skala 1–5. En femma ska vara ovanlig och ges endast när båda kraven är helt uppfyllda.

- relevance_score: 5 endast om källtexten är ett mycket naturligt, direkt och korrekt svar på just instruktionen. 3 betyder delvis relevant; 1 betyder fel, alltför vag eller metatext.
- idiomaticity_score: 5 endast om texten är en enda välformulerad, idiomatisk svensk fråga/instruktion. Sätt 1 om den innehåller resonemang, analys, rubriker, punktlistor, formatinstruktioner, engelska metauttryck eller flera steg i stället för själva instruktionen.

Svara endast med giltig JSON:
{{"relevance_score": <heltal 1-5>, "idiomaticity_score": <heltal 1-5>, "reasoning": "kort motivering på svenska"}}
"""


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE, device_map="auto")
    model.eval()
    return tokenizer, model


def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


def make_inputs(tokenizer, model, prompt):
    messages = [
        {"role": "system", "content": "Ge endast det efterfrågade svaret. Visa aldrig tankegång, analys eller arbetssteg."},
        {"role": "user", "content": prompt},
    ]
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    )
    # Qwen 3.5 templates support disabling visible reasoning. Older templates safely fall back.
    try:
        inputs = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        inputs = tokenizer.apply_chat_template(messages, **kwargs)
    return inputs.to(model.device)


def normalize_instruction(text):
    text = text.strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return re.sub(r"\s+", " ", text).strip().strip('"')


def is_valid_instruction(instruction):
    if not instruction or len(instruction) > 350 or "\n" in instruction:
        return False
    forbidden = ("thinking process", "analysis:", "analys:", "step by step", "<think>", "1.", "**", "input:", "output:")
    request_starts = ("vad ", "vem ", "vilken ", "vilket ", "vilka ", "hur ", "var ", "när ", "beskriv ", "skriv ", "förklara ", "redogör ", "återge ", "berätta ", "ge ", "identifiera ", "sammanfatta ")
    return not any(marker in instruction.lower() for marker in forbidden) and ("?" in instruction or instruction.lower().startswith(request_starts))


def generate_instruction(tokenizer, model, prompt):
    inputs = make_inputs(tokenizer, model, prompt)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.35,
            top_p=0.85,
            repetition_penalty=1.08,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    return normalize_instruction(tokenizer.decode(generated_ids, skip_special_tokens=True))


def judge_instruction(tokenizer, model, seed_doc, instruction):
    # Invalid format receives deterministic failing scores; it must never be rewarded by the LLM judge.
    if not is_valid_instruction(instruction):
        return {
            "relevance_score": 1,
            "idiomaticity_score": 1,
            "reasoning": "Instruktionen är inte en ensam, färdig svensk fråga eller instruktion utan innehåller metatext eller resonemang.",
        }

    inputs = make_inputs(tokenizer, model, JUDGE_PROMPT.format(seed_doc=seed_doc, instruction=instruction))
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=160, do_sample=False)
    response = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    try:
        if "```" in response:
            response = response.split("```")[1].replace("json", "", 1).strip()
        result = json.loads(response)
        relevance = result.get("relevance_score")
        idiomaticity = result.get("idiomaticity_score")
        if not isinstance(relevance, int) or not isinstance(idiomaticity, int):
            raise ValueError("Scores must be integers")
        if not (1 <= relevance <= 5 and 1 <= idiomaticity <= 5):
            raise ValueError("Scores must be from 1 to 5")
        return {"relevance_score": relevance, "idiomaticity_score": idiomaticity, "reasoning": str(result.get("reasoning", ""))}
    except (json.JSONDecodeError, ValueError, TypeError, IndexError) as error:
        return {"relevance_score": 1, "idiomaticity_score": 1, "reasoning": f"Kunde inte tolka domarens svar: {error}"}


def needs_refinement(evaluation):
    return evaluation["relevance_score"] < SCORE_THRESHOLD or evaluation["idiomaticity_score"] < SCORE_THRESHOLD


def passes_threshold(evaluation):
    return evaluation["relevance_score"] >= SCORE_THRESHOLD and evaluation["idiomaticity_score"] >= SCORE_THRESHOLD


if not os.path.exists(SEED_INPUT_PATH):
    raise FileNotFoundError(f"Seed file not found: {SEED_INPUT_PATH}")
with open(SEED_INPUT_PATH, "r", encoding="utf-8") as f:
    seeds = [json.loads(line) for line in f if line.strip()]

records = [{"seed_doc": seed.get("text", "").strip()} for seed in seeds if seed.get("text", "").strip()]


def scores_for_output(evaluation):
    return {
        "relevance": evaluation["relevance_score"],
        "idiomaticity": evaluation["idiomaticity_score"],
    }


def write_snapshot(output_path):
    """Save every pair after this stage; only the current instruction is exposed."""
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            output_record = {
                "seed_doc": record["seed_doc"],
                "generated_instruction": record["current_instruction"],
                "scores": scores_for_output(record["current_evaluation"]),
                "retries": record["retries"],
            }
            f.write(json.dumps(output_record, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to: {output_path}")


# Stage 1: generate and evaluate every seed pair, then checkpoint it.
generator_tokenizer, generator_model = load_model(GENERATOR_MODEL_ID)
for record in tqdm(records, desc="Generating initial instructions with Qwen 3.5"):
    record["current_instruction"] = generate_instruction(
        generator_tokenizer, generator_model, INITIAL_PROMPT.format(seed_doc=record["seed_doc"])
    )
release_model(generator_model)
del generator_model, generator_tokenizer
gc.collect()
torch.cuda.empty_cache()

judge_tokenizer, judge_model = load_model(JUDGE_MODEL_ID)
for record in tqdm(records, desc="Evaluating initial instructions with Qwen 2.5"):
    record["current_evaluation"] = judge_instruction(
        judge_tokenizer, judge_model, record["seed_doc"], record["current_instruction"]
    )
    record["retries"] = []
release_model(judge_model)
del judge_model, judge_tokenizer
gc.collect()
torch.cuda.empty_cache()

write_snapshot(INITIAL_OUTPUT_PATH)


# Later stages regenerate only records where relevance OR idiomaticity is below 3.
for attempt in range(1, MAX_REGENERATION_ATTEMPTS + 1):
    candidates = [record for record in records if needs_refinement(record["current_evaluation"])]
    if not candidates:
        print(f"All records met the threshold after stage {attempt - 1}; no further retry file is needed.")
        break

    generator_tokenizer, generator_model = load_model(GENERATOR_MODEL_ID)
    for record in tqdm(candidates, desc=f"Regenerating retry {attempt} with Qwen 3.5"):
        record["previous_evaluation"] = record["current_evaluation"]
        record["current_instruction"] = generate_instruction(
            generator_tokenizer,
            generator_model,
            REFINEMENT_PROMPT.format(
                seed_doc=record["seed_doc"],
                instruction=record["current_instruction"],
                eval_reasoning=record["previous_evaluation"]["reasoning"],
            ),
        )
    release_model(generator_model)
    del generator_model, generator_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    judge_tokenizer, judge_model = load_model(JUDGE_MODEL_ID)
    for record in tqdm(candidates, desc=f"Evaluating retry {attempt} with Qwen 2.5"):
        retry_evaluation = judge_instruction(
            judge_tokenizer, judge_model, record["seed_doc"], record["current_instruction"]
        )
        record["retries"].append({
            "attempt": attempt,
            "feedback": record["previous_evaluation"]["reasoning"],
            "scores": scores_for_output(retry_evaluation),
        })
        record["current_evaluation"] = retry_evaluation
        del record["previous_evaluation"]
    release_model(judge_model)
    del judge_model, judge_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    write_snapshot(REGENERATED_OUTPUT_TEMPLATE.format(attempt=attempt))

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (28 outputs left out)

Regenerating retry 3 with Qwen 3.5: 100%|██████████| 19/19 [00:29<00:00,  1.55s/it]


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Evaluating retry 3 with Qwen 2.5: 100%|██████████| 19/19 [16:56<00:00, 53.51s/it]


Saved 150 records to: regenerated_instruction_3.jsonl
